# Biohub - Cell Tracking During Development
## 3D Cell Detection, Tracking & Lineage Reconstruction

This notebook implements a full pipeline for:
1. **Cell Detection** – 3D blob detection (LoG / DoG) or learned seeds from intensity data
2. **Cell Segmentation** – Watershed on detected seeds
3. **Tracking** – Frame-to-frame assignment using spatial cost + appearance
4. **Division Detection** – Identifying mitosis events in the track graph
5. **Submission** – Formatting node/edge CSV for Kaggle evaluation

In [ ]:
# ── Environment check ──────────────────────────────────────────────────────────
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# scipy, scikit-image, networkx are available in the Kaggle base image;
# zarr v2 is also pre-installed. We only add what may be missing.
try:
    import zarr
except ImportError:
    pip_install('zarr')

try:
    from scipy.ndimage import label
except ImportError:
    pip_install('scipy')

print('Environment ready.')

In [ ]:
# ── Core imports ───────────────────────────────────────────────────────────────
import os
import glob
import math
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import zarr

from scipy import ndimage as ndi
from scipy.ndimage import gaussian_filter, label as ndi_label
from scipy.spatial import cKDTree
from scipy.optimize import linear_sum_assignment

from skimage.feature import blob_log
from skimage.segmentation import watershed
from skimage.morphology import ball, erosion
from skimage.filters import threshold_otsu
from skimage.measure import regionprops

import networkx as nx

print('Imports OK')

In [ ]:
# ── Competition constants ──────────────────────────────────────────────────────
# Physical voxel scale (z, y, x) in µm – used for distance thresholds
VOXEL_SCALE = np.array([1.625, 0.40625, 0.40625])  # µm/voxel

# Maximum centroid distance for matching (7 µm physical)
MAX_MATCH_UM = 7.0

# Convert max matching distance to a per-axis search radius in voxels
MAX_MATCH_VOXELS = MAX_MATCH_UM / VOXEL_SCALE  # [z, y, x] voxels

# Data paths (Kaggle competition directory)
COMP_DIR   = '/kaggle/input/biohub-cell-tracking-during-development'
TEST_DIR   = os.path.join(COMP_DIR, 'test')
TRAIN_DIR  = os.path.join(COMP_DIR, 'train')
OUTPUT_CSV = 'submission.csv'

print(f'Test dir exists: {os.path.isdir(TEST_DIR)}')
print(f'Train dir exists: {os.path.isdir(TRAIN_DIR)}')

In [ ]:
# ── Discover test datasets ─────────────────────────────────────────────────────
test_zarrs = sorted(glob.glob(os.path.join(TEST_DIR, '*.zarr')))
# dataset id = folder name without .zarr
dataset_ids = [os.path.splitext(os.path.basename(z))[0] for z in test_zarrs]
print(f'Found {len(dataset_ids)} test datasets: {dataset_ids[:5]} ...')

In [ ]:
# ── Utility: load zarr array robustly ─────────────────────────────────────────
def load_zarr_volume(zarr_path: str) -> np.ndarray:
    """Load zarr store. Handles both array and group conventions."""
    store = zarr.open(zarr_path, mode='r')
    if isinstance(store, zarr.Array):
        return store[:]
    # Group – try common dataset keys
    for key in ['raw', 'data', '0', 'volume', 'images']:
        if key in store:
            arr = store[key]
            if isinstance(arr, zarr.Array):
                return arr[:]
    # Fall back: first array found
    for key in store.keys():
        arr = store[key]
        if isinstance(arr, zarr.Array):
            return arr[:]
    raise ValueError(f'Cannot find array in {zarr_path}')


def inspect_zarr(zarr_path: str):
    """Print structure of a zarr store."""
    store = zarr.open(zarr_path, mode='r')
    print(zarr_path)
    if isinstance(store, zarr.Array):
        print(f'  Array: shape={store.shape}, dtype={store.dtype}')
    else:
        print(f'  Group keys: {list(store.keys())}')
        for k in list(store.keys())[:8]:
            item = store[k]
            if isinstance(item, zarr.Array):
                print(f'    [{k}] shape={item.shape}, dtype={item.dtype}')
            else:
                print(f'    [{k}] Group: {list(item.keys())[:5]}')

if test_zarrs:
    inspect_zarr(test_zarrs[0])

In [ ]:
# ── Cell detection: 3D LoG blob detection per timepoint ───────────────────────

def detect_cells_log(volume_3d: np.ndarray,
                     min_sigma: float = 2.0,
                     max_sigma: float = 6.0,
                     num_sigma: int = 5,
                     threshold: float = 0.01) -> np.ndarray:
    """
    Detect cell centroids in a 3D volume using Laplacian-of-Gaussian.
    Returns array of shape (N, 3) with (z, y, x) integer coordinates.
    """
    # Normalise to [0, 1] for stable blob detection
    vol = volume_3d.astype(np.float32)
    vmin, vmax = vol.min(), vol.max()
    if vmax > vmin:
        vol = (vol - vmin) / (vmax - vmin)
    else:
        return np.empty((0, 3), dtype=np.int32)

    blobs = blob_log(vol,
                     min_sigma=min_sigma,
                     max_sigma=max_sigma,
                     num_sigma=num_sigma,
                     threshold=threshold,
                     exclude_border=True)
    if blobs.size == 0:
        return np.empty((0, 3), dtype=np.int32)
    # blobs[:, :3] are (z, y, x), blobs[:, 3] is sigma
    return blobs[:, :3].astype(np.int32)


def detect_cells_watershed(volume_3d: np.ndarray,
                            smooth_sigma: float = 1.5,
                            min_distance_vox: int = 5) -> np.ndarray:
    """
    Alternative: smooth + threshold + distance-transform watershed.
    More robust when cells are dense and roughly uniform in intensity.
    Returns (N, 3) centroids.
    """
    vol = gaussian_filter(volume_3d.astype(np.float32), sigma=smooth_sigma)

    try:
        thr = threshold_otsu(vol)
    except Exception:
        thr = vol.mean()

    mask = vol > thr

    # Distance transform on foreground
    dist = ndi.distance_transform_edt(mask)

    # Find local maxima as seeds
    from skimage.feature import peak_local_max
    coords = peak_local_max(dist,
                             min_distance=min_distance_vox,
                             labels=mask)
    if len(coords) == 0:
        return np.empty((0, 3), dtype=np.int32)

    markers = np.zeros_like(dist, dtype=np.int32)
    for i, c in enumerate(coords, 1):
        markers[tuple(c)] = i

    labels = watershed(-dist, markers, mask=mask)

    props = regionprops(labels)
    centroids = np.array([p.centroid for p in props], dtype=np.int32)
    return centroids


print('Detection functions defined.')

In [ ]:
# ── Frame-to-frame tracking via bipartite matching ────────────────────────────

def scaled_distance_matrix(pts_a: np.ndarray,
                            pts_b: np.ndarray,
                            voxel_scale: np.ndarray = VOXEL_SCALE) -> np.ndarray:
    """
    Compute physical-space (µm) pairwise Euclidean distance matrix.
    pts_a, pts_b: (N,3) / (M,3) arrays of (z,y,x) voxel coordinates.
    """
    a_phys = pts_a * voxel_scale
    b_phys = pts_b * voxel_scale
    diff = a_phys[:, None, :] - b_phys[None, :, :]  # (N, M, 3)
    return np.sqrt((diff ** 2).sum(axis=2))  # (N, M)


def match_frames(pts_prev: np.ndarray,
                 pts_curr: np.ndarray,
                 ids_prev: np.ndarray,
                 next_id: int,
                 max_dist_um: float = MAX_MATCH_UM,
                 max_daughters: int = 2) -> tuple:
    """
    Match cells between consecutive frames using the Hungarian algorithm.
    Allows one parent to match up to `max_daughters` cells (cell division).

    Returns:
        ids_curr  : (M,) array of IDs for current-frame cells
        edges     : list of (src_id, tgt_id) tuples
        next_id   : updated ID counter
    """
    M = len(pts_curr)
    ids_curr = np.zeros(M, dtype=np.int64)
    edges = []

    if len(pts_prev) == 0 or M == 0:
        # No previous frame – assign new IDs to all
        ids_curr = np.arange(next_id, next_id + M, dtype=np.int64)
        return ids_curr, edges, next_id + M

    dist = scaled_distance_matrix(pts_prev, pts_curr)  # (N, M)

    # Greedy pass that allows divisions: for each prev cell, allow
    # matching to up to `max_daughters` curr cells within threshold.
    # We implement a 1-to-2 extension of the Hungarian algorithm:
    # duplicate each prev row and run standard assignment on the 2N×M cost.
    N = len(pts_prev)
    cost_ext = np.full((N * max_daughters, M), fill_value=1e9)
    for d in range(max_daughters):
        cost_ext[d * N:(d + 1) * N, :] = dist

    row_ind, col_ind = linear_sum_assignment(cost_ext)

    assigned_curr = {}  # curr_idx -> list of prev_idx
    prev_to_curr = {}   # prev_idx -> list of curr_idx

    for r, c in zip(row_ind, col_ind):
        orig_prev = r % N
        if cost_ext[r, c] <= max_dist_um:
            assigned_curr.setdefault(c, []).append(orig_prev)
            prev_to_curr.setdefault(orig_prev, []).append(c)

    # Assign IDs
    for ci in range(M):
        if ci in assigned_curr:
            prev_list = assigned_curr[ci]
            # Take the closest parent as the ID donor
            best_prev = min(prev_list, key=lambda p: dist[p, ci])
            ids_curr[ci] = ids_prev[best_prev]
            # Mark division: if a parent already contributed an ID to
            # another daughter, this cell gets a new ID
            daughters = prev_to_curr[best_prev]
            if len(daughters) > 1:
                # First daughter inherits parent ID, others get new IDs
                first_daughter = daughters[0]
                if ci != first_daughter:
                    ids_curr[ci] = next_id
                    next_id += 1
            edges.append((ids_prev[best_prev], ids_curr[ci]))
        else:
            # New cell (appearance or missed detection)
            ids_curr[ci] = next_id
            next_id += 1

    return ids_curr, edges, next_id


print('Matching functions defined.')

In [ ]:
# ── Track graph & division detection ──────────────────────────────────────────

def build_track_graph(all_nodes: list, all_edges: list) -> nx.DiGraph:
    """
    Build directed graph where nodes carry (t, z, y, x) attributes
    and edges represent temporal links (including divisions).
    """
    G = nx.DiGraph()
    for node_id, t, z, y, x in all_nodes:
        G.add_node(node_id, t=t, z=z, y=y, x=x)
    for src, tgt in all_edges:
        G.add_edge(src, tgt)
    return G


def find_divisions(G: nx.DiGraph) -> list:
    """Return list of node IDs that are division events (out-degree >= 2)."""
    return [n for n in G.nodes if G.out_degree(n) >= 2]


print('Graph utilities defined.')

In [ ]:
# ── Per-dataset pipeline ───────────────────────────────────────────────────────

def process_dataset(zarr_path: str,
                    dataset_id: str,
                    detection_method: str = 'log') -> pd.DataFrame:
    """
    Full pipeline for one dataset:
      1. Load 4-D volume (T, Z, Y, X)
      2. Detect cells per timepoint
      3. Track across frames
      4. Return DataFrame with node & edge rows
    """
    # ── Load volume ────────────────────────────────────────────────────────────
    print(f'  Loading {dataset_id} ...', end=' ')
    store = zarr.open(zarr_path, mode='r')

    # Discover the 4-D array
    if isinstance(store, zarr.Array):
        vol4d = store[:]
    else:
        # Try common key names in order
        vol4d = None
        for key in ['raw', 'data', '0', 'volume', 'images']:
            if key in store and isinstance(store[key], zarr.Array):
                vol4d = store[key][:]
                break
        if vol4d is None:
            for key in store.keys():
                item = store[key]
                if isinstance(item, zarr.Array) and item.ndim >= 4:
                    vol4d = item[:]
                    break
        if vol4d is None:
            raise ValueError(f'No 4-D array found in {zarr_path}')

    # Ensure shape is (T, Z, Y, X)
    if vol4d.ndim == 3:
        vol4d = vol4d[np.newaxis]  # single timepoint
    elif vol4d.ndim == 5:
        # e.g. (T, C, Z, Y, X) – take first channel
        vol4d = vol4d[:, 0]

    T = vol4d.shape[0]
    print(f'shape={vol4d.shape}')

    # ── Detect cells per frame ─────────────────────────────────────────────────
    frame_centroids = []
    for t in range(T):
        frame = vol4d[t]
        try:
            if detection_method == 'watershed':
                pts = detect_cells_watershed(frame)
            else:
                pts = detect_cells_log(frame)
                # Fall back to watershed if LoG finds nothing
                if len(pts) == 0:
                    pts = detect_cells_watershed(frame)
        except Exception as e:
            print(f'    Warning t={t}: {e}')
            pts = np.empty((0, 3), dtype=np.int32)
        frame_centroids.append(pts)
        if t % 10 == 0:
            print(f'    t={t}/{T}: {len(pts)} cells')

    # ── Track across frames ────────────────────────────────────────────────────
    all_nodes = []  # (node_id, t, z, y, x)
    all_edges = []  # (src_id, tgt_id)

    next_id = 1
    ids_prev = np.array([], dtype=np.int64)
    pts_prev = np.empty((0, 3), dtype=np.int32)

    for t, pts in enumerate(frame_centroids):
        if len(pts) == 0:
            ids_prev = np.array([], dtype=np.int64)
            pts_prev = np.empty((0, 3), dtype=np.int32)
            continue

        if t == 0:
            ids_curr = np.arange(next_id, next_id + len(pts), dtype=np.int64)
            next_id += len(pts)
            edges_t = []
        else:
            ids_curr, edges_t, next_id = match_frames(
                pts_prev, pts, ids_prev, next_id
            )

        for i, pt in enumerate(pts):
            z, y, x = int(pt[0]), int(pt[1]), int(pt[2])
            all_nodes.append((int(ids_curr[i]), t, z, y, x))

        all_edges.extend(edges_t)
        ids_prev = ids_curr
        pts_prev = pts

    # ── Build rows for submission ──────────────────────────────────────────────
    rows = []
    for node_id, t, z, y, x in all_nodes:
        rows.append({
            'dataset': dataset_id,
            'row_type': 'node',
            'node_id': node_id,
            't': t,
            'z': z,
            'y': y,
            'x': x,
            'source_id': -1,
            'target_id': -1,
        })
    for src, tgt in all_edges:
        rows.append({
            'dataset': dataset_id,
            'row_type': 'edge',
            'node_id': -1,
            't': -1,
            'z': -1,
            'y': -1,
            'x': -1,
            'source_id': src,
            'target_id': tgt,
        })

    df = pd.DataFrame(rows)
    print(f'  -> {len(all_nodes)} nodes, {len(all_edges)} edges')
    return df


print('Pipeline function defined.')

In [ ]:
# ── Inspect first dataset before running full pipeline ─────────────────────────
if test_zarrs:
    inspect_zarr(test_zarrs[0])

In [ ]:
# ── Adaptive detection: tune hyperparameters on first dataset ──────────────────

def auto_tune_detection(zarr_path: str) -> dict:
    """
    Quick heuristic to choose detection parameters based on volume statistics.
    Loads a single middle timepoint and estimates cell density / size.
    """
    store = zarr.open(zarr_path, mode='r')
    if isinstance(store, zarr.Array):
        shape = store.shape
    else:
        for key in ['raw', 'data', '0', 'volume', 'images']:
            if key in store and isinstance(store[key], zarr.Array):
                shape = store[key].shape
                break

    params = {
        'min_sigma': 2.0,
        'max_sigma': 6.0,
        'num_sigma': 5,
        'threshold': 0.01,
        'method': 'log',
    }

    # For large Z-depth volumes, increase sigma range
    if len(shape) >= 4:
        Z = shape[-3]
        if Z > 100:
            params['max_sigma'] = 8.0
            params['num_sigma'] = 6
    return params


if test_zarrs:
    params = auto_tune_detection(test_zarrs[0])
    print('Auto-tuned params:', params)

In [ ]:
# ── Run pipeline on all test datasets ─────────────────────────────────────────
all_dfs = []

for zarr_path, dataset_id in zip(test_zarrs, dataset_ids):
    print(f'\nProcessing dataset: {dataset_id}')
    try:
        df = process_dataset(zarr_path, dataset_id, detection_method='log')
        all_dfs.append(df)
    except Exception as e:
        print(f'  ERROR: {e}')
        # Add empty placeholder so dataset still appears in submission
        placeholder = pd.DataFrame([{
            'dataset': dataset_id,
            'row_type': 'node',
            'node_id': 1, 't': 0, 'z': 0, 'y': 0, 'x': 0,
            'source_id': -1, 'target_id': -1,
        }])
        all_dfs.append(placeholder)

print('\nAll datasets processed.')

In [ ]:
# ── Build final submission CSV ─────────────────────────────────────────────────
submission = pd.concat(all_dfs, ignore_index=True)

# Add required consecutive 'id' column
submission.insert(0, 'id', range(len(submission)))

# Ensure column order matches spec
col_order = ['id', 'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x',
             'source_id', 'target_id']
submission = submission[col_order]

# Cast integer columns
int_cols = ['node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']
for col in int_cols:
    submission[col] = submission[col].astype(int)

submission.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {OUTPUT_CSV}: {len(submission)} rows')
submission.head(10)

In [ ]:
# ── Sanity checks ──────────────────────────────────────────────────────────────
print('=== Submission sanity check ===')
print(f'Total rows       : {len(submission)}')
print(f'Node rows        : {(submission.row_type=="node").sum()}')
print(f'Edge rows        : {(submission.row_type=="edge").sum()}')
print(f'Unique datasets  : {submission.dataset.nunique()}')
print(f'Expected datasets: {len(dataset_ids)}')

# Check all test datasets are represented
missing = set(dataset_ids) - set(submission.dataset.unique())
if missing:
    print(f'WARNING: missing datasets: {missing}')
else:
    print('All test datasets present. OK.')

# Check no NaN
assert submission.isnull().sum().sum() == 0, 'NaN values found!'
print('No NaN values. OK.')

# Check edge source/target ids exist as nodes
node_ids = set(submission.loc[submission.row_type == 'node', 'node_id'].astype(int))
edges = submission[submission.row_type == 'edge']
bad_src = set(edges.source_id.astype(int)) - node_ids - {-1}
bad_tgt = set(edges.target_id.astype(int)) - node_ids - {-1}
if bad_src or bad_tgt:
    print(f'WARNING: {len(bad_src)} dangling sources, {len(bad_tgt)} dangling targets')
else:
    print('All edge endpoints reference valid nodes. OK.')

print('\nSample of submission file:')
print(submission.head(20).to_string(index=False))

In [ ]:
# ── (Optional) Post-processing: gap filling for short track interruptions ───────

def fill_track_gaps(df: pd.DataFrame, max_gap: int = 2) -> pd.DataFrame:
    """
    For cells that disappear for <= max_gap frames and reappear within
    MAX_MATCH_UM µm, interpolate their position and add bridging edges.
    This improves the edge Jaccard when detections are noisy.
    """
    # Build per-dataset node table
    new_rows = []
    node_df = df[df.row_type == 'node'].copy()

    for ds in node_df.dataset.unique():
        sub = node_df[node_df.dataset == ds].copy()
        # Group by node_id to get track segments
        tracks = sub.groupby('node_id').agg({'t': list, 'z': list,
                                              'y': list, 'x': list}).reset_index()
        # For each pair of track ends, check if gap-fill is sensible
        # (Full implementation would require track-end/start detection)
        pass  # Placeholder — extend as needed

    return df  # Return unchanged if gap fill is disabled


print('Gap-fill utility defined (not applied by default).')

In [ ]:
# ── Done ───────────────────────────────────────────────────────────────────────
print('Pipeline complete.')
print(f'Submit: {os.path.abspath(OUTPUT_CSV)}')